The Required Files are avaliable in the Google Drive which can be accessed through Readme file in Github Repository

In [55]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, accuracy_score, precision_score,
                            recall_score, f1_score, roc_auc_score, roc_curve, auc)
from sklearn.base import BaseEstimator, ClassifierMixin
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import time
import datetime
import os
from google.colab import drive

In [56]:
#The Required Files are avaliable in the Google Drive which can be accessed through Readme file in Github Repository
# Data directory - using the processed data path
PROCESSED_DATA_PATH = 'nih_chestxray_augmented.csv' #nih_chestxray_augmented.csv can be found in the google drive link in the readme file
OUTPUT_DIR = 'randomforest_results'

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Set random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [57]:
class CustomRandomForestClassifier(BaseEstimator, ClassifierMixin):
    """Custom multi-label classifier using Random Forest."""
    def __init__(self, class_weights=None, **rf_params):
        # Default Random Forest parameters
        self.default_params = {
            'n_estimators': 200,
            'max_depth': 20,
            'min_samples_split': 5,
            'min_samples_leaf': 2,
            'max_features': 'sqrt',
            'bootstrap': True,
            'n_jobs': -1,
            'random_state': RANDOM_SEED,
            'verbose': 1
        }

        # Update with user params
        for k, v in rf_params.items():
            self.default_params[k] = v

        self.rf_params = self.default_params
        self.class_weights = class_weights
        self.estimators_ = []

    def fit(self, X, y):
        """Fits a separate RandomForest classifier for each label."""
        self.estimators_ = []

        # Use tqdm for progress tracking
        for i, class_label in enumerate(tqdm(y.columns, desc="Training Random Forest models", unit="class")):
            print(f"\nTraining Random Forest for {class_label} ({i+1}/{len(y.columns)})...")

            # Add class weight for this specific label
            params = self.rf_params.copy()
            if self.class_weights is not None and class_label in self.class_weights:
                class_weight_dict = {0: 1.0, 1: self.class_weights[class_label]}
                params['class_weight'] = class_weight_dict
                print(f"  - Using class_weight = {class_weight_dict}")

            # Create and train estimator
            estimator = RandomForestClassifier(**params)

            # Training progress tracking
            class_time_start = time.time()
            estimator.fit(X, y.iloc[:, i])

            # Report training completion
            class_time = time.time() - class_time_start
            print(f"  - Trained in {class_time:.2f} seconds ({class_time/60:.2f} minutes)")

            self.estimators_.append(estimator)

            # Display feature importance
            if hasattr(estimator, 'feature_importances_'):
                importances = estimator.feature_importances_
                indices = np.argsort(importances)[-5:]  # Top 5 features
                top_features = [X.columns[i] for i in indices[::-1]]
                print(f"  - Top features: {', '.join(top_features)}")

        return self

    def predict(self, X):
        """Make predictions for all labels."""
        n_samples = X.shape[0]
        n_labels = len(self.estimators_)
        y_pred = np.zeros((n_samples, n_labels), dtype=int)

        for i, estimator in enumerate(self.estimators_):
            y_pred[:, i] = estimator.predict(X)

        return y_pred

    def predict_proba(self, X):
        """Make probability predictions for all labels."""
        n_samples = X.shape[0]
        n_labels = len(self.estimators_)
        y_pred_proba = []

        for i, estimator in enumerate(self.estimators_):
            if hasattr(estimator, 'predict_proba'):
                y_pred_proba.append(estimator.predict_proba(X))
            else:
                # Fallback for estimators without predict_proba
                preds = estimator.predict(X)
                proba = np.zeros((n_samples, 2))
                proba[:, 1] = preds
                proba[:, 0] = 1 - preds
                y_pred_proba.append(proba)

        return y_pred_proba

In [58]:
def limit_samples_per_class(X, y, max_samples=2000):
    """
    Limit the number of samples per class to max_samples while maintaining ratio.

    Parameters:
    -----------
    X : DataFrame
        Feature data
    y : DataFrame
        Label data
    max_samples : int
        Maximum number of samples per class

    Returns:
    --------
    X_balanced : DataFrame
        Balanced feature data
    y_balanced : DataFrame
        Balanced label data
    """
    print(f"Limiting samples per class to maximum {max_samples}...")

    # Get the current counts for each class
    class_counts = y.sum().to_dict()
    print("Original class counts:")
    for label, count in sorted(class_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"  - {label}: {count}")

    # Dictionary to store indices to keep for each class
    indices_by_class = {}

    # For each class, identify indices to keep
    for label in y.columns:
        pos_indices = np.where(y[label] == 1)[0]

        if len(pos_indices) <= max_samples:
            # Keep all samples if already under limit
            indices_by_class[label] = pos_indices
            print(f"  - {label}: Keeping all {len(pos_indices)} samples (under max limit)")
        else:
            # Randomly select samples to keep
            np.random.seed(RANDOM_SEED + hash(label) % 1000)  # Different seed for each label
            selected_indices = np.random.choice(pos_indices, size=max_samples, replace=False)
            indices_by_class[label] = selected_indices
            print(f"  - {label}: Reduced from {len(pos_indices)} to {max_samples} samples")

    # Combine all indices to keep across all labels
    all_indices_to_keep = set()
    for indices in indices_by_class.values():
        all_indices_to_keep.update(indices)

    # Convert to list and sort
    all_indices_to_keep = sorted(list(all_indices_to_keep))

    # Create balanced dataset
    X_balanced = X.iloc[all_indices_to_keep].reset_index(drop=True)
    y_balanced = y.iloc[all_indices_to_keep].reset_index(drop=True)

    # Print the resulting class distribution
    new_class_counts = y_balanced.sum().to_dict()
    print("\nNew class counts after limiting to maximum of", max_samples, "samples per class:")
    for label, count in sorted(new_class_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"  - {label}: {count} ({count/len(y_balanced)*100:.2f}%)")

    print(f"\nTotal samples after limiting: {len(X_balanced)} (reduced from {len(X)})")

    return X_balanced, y_balanced

In [59]:
def stratified_multilabel_split(X, y, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1):
    """
    Perform stratified split ensuring consistent class ratios across all splits.

    Parameters:
    -----------
    X : DataFrame
        Feature data
    y : DataFrame
        Label data
    train_ratio, val_ratio, test_ratio : float
        Ratio for train, validation, and test splits

    Returns:
    --------
    X_train, X_val, X_test, y_train, y_val, y_test
    """
    print("\nPerforming stratified split with ratios - Train: 8, Val: 1, Test: 1")
    print("This maintains the same class ratios across all splits.")

    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-10, "Split ratios must sum to 1"

    # For each unique label combination, we'll split proportionally
    # First, get binary representation of each sample's labels
    y_binary = y.astype(int).astype(str).apply(''.join, axis=1)

    # Group samples by label combination
    combo_groups = {}
    for i, combo in enumerate(y_binary):
        if combo not in combo_groups:
            combo_groups[combo] = []
        combo_groups[combo].append(i)

    print(f"Found {len(combo_groups)} unique label combinations")

    # Prepare containers for split indices
    train_indices = []
    val_indices = []
    test_indices = []

    # For each combination, apply the split ratios
    for combo, indices in tqdm(combo_groups.items(), desc="Splitting data by label combination"):
        if len(indices) == 0:
            continue

        # Determine which labels are present in this combination
        present_labels = []
        for i, char in enumerate(combo):
            if char == '1':
                present_labels.append(y.columns[i])

        if len(present_labels) > 0:
            labels_str = ", ".join(present_labels)
        else:
            labels_str = "No labels (all zeros)"

        print(f"Splitting combination with labels: {labels_str} ({len(indices)} samples)")

        # Ensure reproducibility
        np.random.seed(RANDOM_SEED)
        np.random.shuffle(indices)

        # Calculate indices for each split
        n_total = len(indices)
        n_train = int(n_total * train_ratio)
        n_val = int(n_total * val_ratio)

        # Ensure at least 1 sample per split if possible
        if n_total >= 3:
            # Standard case - enough samples for all splits
            train_idx = indices[:n_train]
            val_idx = indices[n_train:n_train+n_val]
            test_idx = indices[n_train+n_val:]
            print(f"  - Split: Train {len(train_idx)}, Val {len(val_idx)}, Test {len(test_idx)}")
        elif n_total == 2:
            # Only 2 samples - prioritize train and validation
            train_idx = [indices[0]]
            val_idx = [indices[1]]
            test_idx = []
            print(f"  - Only 2 samples: Train 1, Val 1, Test 0")
        else:  # n_total == 1
            # Only 1 sample - place in training set
            train_idx = indices
            val_idx = []
            test_idx = []
            print(f"  - Only 1 sample: Train 1, Val 0, Test 0")

        # Add to our index lists
        train_indices.extend(train_idx)
        val_indices.extend(val_idx)
        test_indices.extend(test_idx)

    # Create the datasets
    X_train = X.iloc[train_indices].reset_index(drop=True)
    y_train = y.iloc[train_indices].reset_index(drop=True)

    X_val = X.iloc[val_indices].reset_index(drop=True)
    y_val = y.iloc[val_indices].reset_index(drop=True)

    X_test = X.iloc[test_indices].reset_index(drop=True)
    y_test = y.iloc[test_indices].reset_index(drop=True)

    # Print split information
    print(f"\nSplit results:")
    print(f"  - Training set: {len(X_train)} samples ({len(X_train)/len(X)*100:.1f}%)")
    print(f"  - Validation set: {len(X_val)} samples ({len(X_val)/len(X)*100:.1f}%)")
    print(f"  - Test set: {len(X_test)} samples ({len(X_test)/len(X)*100:.1f}%)")

    # Validate that class ratios are preserved
    print("\nVerifying class distribution across splits:")
    for label in y.columns:
        total = y[label].sum()
        train = y_train[label].sum()
        val = y_val[label].sum()
        test = y_test[label].sum()

        train_pct = train/total*100 if total > 0 else 0
        val_pct = val/total*100 if total > 0 else 0
        test_pct = test/total*100 if total > 0 else 0

        print(f"  - {label}:")
        print(f"    * Total: {total} samples")
        print(f"    * Train: {train} samples ({train_pct:.1f}%)")
        print(f"    * Val: {val} samples ({val_pct:.1f}%)")
        print(f"    * Test: {test} samples ({test_pct:.1f}%)")

        # Check if the distribution is close to the expected ratios
        expected_train_pct = train_ratio * 100
        expected_val_pct = val_ratio * 100
        expected_test_pct = test_ratio * 100

        print(f"    * Expected ratio - Train: {expected_train_pct:.1f}%, Val: {expected_val_pct:.1f}%, Test: {expected_test_pct:.1f}%")

    return X_train, X_val, X_test, y_train, y_val, y_test

In [60]:
def find_optimal_thresholds(model, X_val, y_val):
    """Find optimal decision thresholds for each class using F1 score."""
    thresholds = {}

    # More granular threshold search
    threshold_range = np.arange(0.05, 0.95, 0.025)

    for i, label in enumerate(tqdm(y_val.columns,
                                  total=len(y_val.columns),
                                  desc="Optimizing thresholds",
                                  unit="label")):
        y_true = y_val.iloc[:, i]

        # Skip if all examples are the same class
        if y_true.nunique() <= 1:
            thresholds[label] = 0.5
            print(f"{label}: Using default threshold = 0.500 (insufficient class variety)")
            continue

        # Get predicted probabilities from the model
        y_proba = model.predict_proba(X_val)[i][:, 1]

        # Try different thresholds
        best_f1 = 0
        best_threshold = 0.5

        # Track all threshold results for plotting
        threshold_results = []

        for threshold in threshold_range:
            y_pred = (y_proba >= threshold).astype(int)
            f1 = f1_score(y_true, y_pred, zero_division=0)
            threshold_results.append((threshold, f1))

            if f1 > best_f1:
                best_f1 = f1
                best_threshold = threshold

        thresholds[label] = best_threshold
        print(f"{label}: optimal threshold = {best_threshold:.3f}, F1 = {best_f1:.4f}")

        # Plot threshold curves for each label
        if threshold_results:
            plt.figure(figsize=(10, 5))
            thresholds_list, f1_scores = zip(*threshold_results)
            plt.plot(thresholds_list, f1_scores, 'b-', marker='o', markersize=4)
            plt.axvline(x=best_threshold, color='r', linestyle='--',
                       label=f'Best threshold: {best_threshold:.3f}')
            plt.title(f'F1 Score vs Threshold for {label}')
            plt.xlabel('Threshold')
            plt.ylabel('F1 Score')
            plt.grid(True)
            plt.legend()
            plt.savefig(f'{OUTPUT_DIR}/threshold_curve_{label}.png')
            plt.close()

    return thresholds

In [61]:
def predict_with_thresholds(model, X, thresholds):
    """Make predictions using custom thresholds."""
    y_pred_proba = model.predict_proba(X)
    n_samples = X.shape[0]
    n_labels = len(model.estimators_)
    y_pred = np.zeros((n_samples, n_labels), dtype=int)

    for i, label in enumerate(tqdm(thresholds.keys(),
                                  total=len(model.estimators_),
                                  desc="Generating predictions",
                                  unit="class")):
        # Get predicted probabilities
        y_proba = y_pred_proba[i][:, 1]

        # Apply custom threshold
        threshold = thresholds[label]
        y_pred[:, i] = (y_proba >= threshold).astype(int)

    return y_pred, y_pred_proba

In [62]:
def calculate_auc_roc(y_true, y_pred_proba, labels):
    """Calculate AUC-ROC for each label and plot ROC curves."""
    auc_scores = {}

    plt.figure(figsize=(12, 10))

    for i, label in enumerate(labels):
        y_true_label = y_true.iloc[:, i]

        # Skip labels with only one class
        if len(np.unique(y_true_label)) <= 1:
            print(f"Warning: Cannot calculate AUC for {label} - not enough class variety")
            continue

        # Get probabilities for positive class
        y_proba = y_pred_proba[i][:, 1]

        try:
            # Calculate ROC curve
            fpr, tpr, _ = roc_curve(y_true_label, y_proba)
            roc_auc = auc(fpr, tpr)
            auc_scores[label] = roc_auc

            # Plot ROC curve
            plt.plot(fpr, tpr, lw=1.5, label=f'{label} (AUC = {roc_auc:.2f})')
        except Exception as e:
            print(f"Error calculating AUC for {label}: {e}")
            continue

    # Plot diagonal line
    plt.plot([0, 1], [0, 1], 'k--', lw=1)

    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves for Multi-label Classification')
    plt.legend(loc="lower right", fontsize='small')
    plt.grid(True, alpha=0.3)
    plt.savefig(f'{OUTPUT_DIR}/roc_curves.png', dpi=300)
    plt.close()

    return auc_scores

In [63]:
def plot_metrics_summary(metrics_df, metric_name, output_path):
    """Plot summary metrics for all classes."""
    plt.figure(figsize=(12, 8))

    # Sort by value
    sorted_df = metrics_df.sort_values(by=metric_name, ascending=False)

    # Create bar plot
    ax = sns.barplot(x=metric_name, y='Label', data=sorted_df, palette='viridis')

    # Add value labels
    for i, v in enumerate(sorted_df[metric_name]):
        if isinstance(v, (int, float)):
            ax.text(max(0.01, v - 0.1), i, f"{v:.3f}", va='center')

    plt.title(f'{metric_name} for each disease class')
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()

In [64]:
def plot_metrics_summary(metrics_df, metric_name, output_path):
    """Plot summary metrics for all classes."""
    plt.figure(figsize=(12, 8))

    # Sort by value
    sorted_df = metrics_df.sort_values(by=metric_name, ascending=False)

    # Create bar plot
    ax = sns.barplot(x=metric_name, y='Label', data=sorted_df, palette='viridis')

    # Add value labels
    for i, v in enumerate(sorted_df[metric_name]):
        if isinstance(v, (int, float)):
            ax.text(max(0.01, v - 0.1), i, f"{v:.3f}", va='center')

    plt.title(f'{metric_name} for each disease class')
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()

In [65]:
def run_randomforest_implementation():
    """Run Random Forest implementation with processed data, class limiting, and 8:1:1 splitting."""
    start_time = time.time()

    print("="*100)
    print(f"NIH CHEST X-RAY CLASSIFICATION WITH RANDOM FOREST (2000 SAMPLES PER CLASS)")
    print(f"Started at: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*100)

    # Step 1: Load the augmented data
    print("\nStep 1: Loading augmented data...")

    # Load the complete dataset
    df = pd.read_csv(PROCESSED_DATA_PATH)
    print(f"Loaded dataset with {len(df)} rows and {df.shape[1]} columns")

    # Separate features, labels, and metadata
    # Identify columns by type
    image_id_column = 'Image Index'
    label_columns = [col for col in df.columns if col in ['Fibrosis', 'Hernia', 'Nodule', 'Consolidation', 'No Finding',
                                                          'Atelectasis', 'Pneumonia', 'Cardiomegaly', 'Emphysema',
                                                          'Infiltration', 'Pleural_Thickening', 'Mass',
                                                          'Effusion', 'Pneumothorax', 'Edema']]
    feature_columns = [col for col in df.columns if col.startswith('feature_')]

    # Extract features and labels
    X = df[feature_columns]
    y = df[label_columns]

    # Display class distribution
    print("\nOriginal class distribution:")
    for label in label_columns:
        count = y[label].sum()
        print(f"  - {label}: {count} samples ({count/len(y)*100:.2f}%)")

    # Step 2: Limit samples per class to 2000
    print("\nStep 2: Limiting samples per class to 2000...")
    X_limited, y_limited = limit_samples_per_class(X, y, max_samples=2000)

    # Step 3: Split the data with 8:1:1 ratio while maintaining class distribution
    print("\nStep 3: Splitting data with 8:1:1 ratio...")
    X_train, X_val, X_test, y_train, y_val, y_test = stratified_multilabel_split(
        X_limited, y_limited, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1
    )

    # Step 4: Calculate class weights
    print("\nStep 4: Calculating class weights...")
    class_weights = {}

    for label in y_train.columns:
        num_neg = (y_train[label] == 0).sum()
        num_pos = (y_train[label] == 1).sum()
        weight = num_neg / max(1, num_pos)
        class_weights[label] = weight
        print(f"  - {label}: {num_pos} positives, {num_neg} negatives, weight = {weight:.2f}")

    # Step 5: Train Random Forest model
    print("\nStep 5: Training Random Forest models...")
    train_start = time.time()

    model = CustomRandomForestClassifier(class_weights=class_weights)
    model.fit(X_train, y_train)

    train_time = time.time() - train_start
    print(f"Training completed in {train_time:.2f} seconds ({train_time/60:.2f} minutes)")

    # Step 6: Optimize thresholds
    print("\nStep 6: Optimizing decision thresholds...")
    thresholds = find_optimal_thresholds(model, X_val, y_val)

    # Step 7: Evaluate on test set
    print("\nStep 7: Evaluating model on test set...")

    # Generate standard predictions (threshold = 0.5)
    print("Generating standard threshold predictions...")
    y_pred_standard = model.predict(X_test)
    y_pred_standard_df = pd.DataFrame(y_pred_standard, columns=y_test.columns, index=y_test.index)

    # Generate optimized threshold predictions
    print("Generating optimized threshold predictions...")
    y_pred_optimized, y_pred_proba = predict_with_thresholds(model, X_test, thresholds)
    y_pred_optimized_df = pd.DataFrame(y_pred_optimized, columns=y_test.columns, index=y_test.index)

    # Calculate AUC-ROC scores
    print("\nCalculating AUC-ROC scores...")
    auc_scores = calculate_auc_roc(y_test, y_pred_proba, y_test.columns)

    # Calculate detailed metrics
    print("\nCalculating detailed metrics...")
    detailed_metrics = []

    # Calculate metrics for each class
    for i, label in enumerate(y_test.columns):
        # Get true and predicted values
        y_true = y_test.iloc[:, i]
        y_pred_std = y_pred_standard[:, i]
        y_pred_opt = y_pred_optimized[:, i]

        # Calculate standard threshold metrics
        accuracy_std = accuracy_score(y_true, y_pred_std)
        precision_std = precision_score(y_true, y_pred_std, zero_division=0)
        recall_std = recall_score(y_true, y_pred_std, zero_division=0)
        f1_std = f1_score(y_true, y_pred_std, zero_division=0)

        # Calculate optimized threshold metrics
        accuracy_opt = accuracy_score(y_true, y_pred_opt)
        precision_opt = precision_score(y_true, y_pred_opt, zero_division=0)
        recall_opt = recall_score(y_true, y_pred_opt, zero_division=0)
        f1_opt = f1_score(y_true, y_pred_opt, zero_division=0)

        # Get AUC score if available
        auc_score = auc_scores.get(label, None)

        # Store metrics
        detailed_metrics.append({
            'Label': label,
            'Support': y_true.sum(),
            'Accuracy (std)': accuracy_std,
            'Precision (std)': precision_std,
            'Recall (std)': recall_std,
            'F1 (std)': f1_std,
            'Accuracy (opt)': accuracy_opt,
            'Precision (opt)': precision_opt,
            'Recall (opt)': recall_opt,
            'F1 (opt)': f1_opt,
            'AUC': auc_score
        })

    # Create metrics DataFrame
    metrics_df = pd.DataFrame(detailed_metrics)

    # Print standard classification report
    print("\nStandard threshold (0.5) metrics:")
    print(classification_report(y_test, y_pred_standard_df))

    # Print optimized classification report
    print("\nOptimized threshold metrics:")
    print(classification_report(y_test, y_pred_optimized_df))

    # Print detailed metrics
    print("\nDetailed metrics for each disease:")
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    print(metrics_df)

    # Step 8: Plot metrics visualizations
    print("\nStep 8: Generating metrics visualizations...")

    # Plot F1 score comparison
    plt.figure(figsize=(14, 10))
    metrics_df_sorted = metrics_df.sort_values('F1 (opt)', ascending=False)

    bar_width = 0.35
    x = np.arange(len(metrics_df_sorted))

    fig, ax = plt.subplots(figsize=(14, 10))
    std_bars = ax.bar(x - bar_width/2, metrics_df_sorted['F1 (std)'], bar_width, label='Standard Threshold', color='blue', alpha=0.7)
    opt_bars = ax.bar(x + bar_width/2, metrics_df_sorted['F1 (opt)'], bar_width, label='Optimized Threshold', color='red', alpha=0.7)

    ax.set_xlabel('Disease Class')
    ax.set_ylabel('F1 Score')
    ax.set_title('F1 Score Comparison: Standard vs. Optimized Thresholds')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics_df_sorted['Label'], rotation=45, ha='right')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/f1_score_comparison.png', dpi=300)
    plt.close()

    # Plot AUC values
    valid_auc = metrics_df[metrics_df['AUC'].notna()].sort_values('AUC', ascending=False)

    plt.figure(figsize=(12, 8))
    plt.barh(valid_auc['Label'], valid_auc['AUC'])
    plt.xlabel('AUC-ROC Score')
    plt.title('AUC-ROC Scores by Disease Class')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/auc_scores.png', dpi=300)
    plt.close()

    # Plot additional metrics
    plot_metrics_summary(metrics_df, 'Precision (opt)', f'{OUTPUT_DIR}/precision_summary.png')
    plot_metrics_summary(metrics_df, 'Recall (opt)', f'{OUTPUT_DIR}/recall_summary.png')
    plot_metrics_summary(metrics_df, 'Accuracy (opt)', f'{OUTPUT_DIR}/accuracy_summary.png')

    # Step 9: Save results and model
    print("\nStep 9: Saving results and model...")

    # Save metrics dataframe
    metrics_df.to_csv(f'{OUTPUT_DIR}/detailed_metrics.csv', index=False)

    # Save model
    with open(f'{OUTPUT_DIR}/randomforest_model.pkl', 'wb') as f:
        pickle.dump(model, f)

    # Save thresholds
    with open(f'{OUTPUT_DIR}/optimal_thresholds.pkl', 'wb') as f:
        pickle.dump(thresholds, f)

    # Save split data information
    split_info = pd.DataFrame({
        'Set': ['Train', 'Validation', 'Test'],
        'Samples': [len(X_train), len(X_val), len(X_test)],
        'Percentage': [len(X_train)/len(X_limited)*100,
                      len(X_val)/len(X_limited)*100,
                      len(X_test)/len(X_limited)*100]
    })
    split_info.to_csv(f'{OUTPUT_DIR}/data_split_info.csv', index=False)

    # Calculate execution time
    total_time = time.time() - start_time

    # Print completion message
    print("\n"+"="*100)
    print(f"RANDOM FOREST IMPLEMENTATION COMPLETED SUCCESSFULLY!")
    print(f"Total execution time: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")
    print(f"Finished at: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*100)

    return {
        'model': model,
        'thresholds': thresholds,
        'metrics': metrics_df
    }

In [66]:
if __name__ == "__main__":
    results = run_randomforest_implementation()
    print("Random Forest implementation completed!")

NIH CHEST X-RAY CLASSIFICATION WITH RANDOM FOREST (2000 SAMPLES PER CLASS)
Started at: 2025-04-22 23:21:07

Step 1: Loading augmented data...
Loaded dataset with 97257 rows and 1042 columns

Original class distribution:
  - Fibrosis: 6520 samples (6.70%)
  - Hernia: 2287 samples (2.35%)
  - Nodule: 17837 samples (18.34%)
  - Consolidation: 16252 samples (16.71%)
  - No Finding: 9133 samples (9.39%)
  - Atelectasis: 29928 samples (30.77%)
  - Pneumonia: 6754 samples (6.94%)
  - Cardiomegaly: 9135 samples (9.39%)
  - Emphysema: 8875 samples (9.13%)
  - Infiltration: 43274 samples (44.49%)
  - Pleural_Thickening: 12917 samples (13.28%)
  - Mass: 18016 samples (18.52%)
  - Effusion: 34856 samples (35.84%)
  - Pneumothorax: 15241 samples (15.67%)
  - Edema: 8861 samples (9.11%)

Step 2: Limiting samples per class to 2000...
Limiting samples per class to maximum 2000...
Original class counts:
  - Infiltration: 43274
  - Effusion: 34856
  - Atelectasis: 29928
  - Mass: 18016
  - Nodule: 17837

Splitting data by label combination:   0%|          | 0/2336 [00:00<?, ?it/s]

Splitting combination with labels: Hernia (405 samples)
  - Split: Train 324, Val 40, Test 41
Splitting combination with labels: Hernia, Infiltration (71 samples)
  - Split: Train 56, Val 7, Test 8
Splitting combination with labels: Infiltration (404 samples)
  - Split: Train 323, Val 40, Test 41
Splitting combination with labels: Emphysema, Pneumothorax (137 samples)
  - Split: Train 109, Val 13, Test 15
Splitting combination with labels: Pleural_Thickening (175 samples)
  - Split: Train 140, Val 17, Test 18
Splitting combination with labels: Emphysema, Infiltration, Effusion, Pneumothorax (39 samples)
  - Split: Train 31, Val 3, Test 5
Splitting combination with labels: Pneumonia, Effusion, Pneumothorax (13 samples)
  - Split: Train 10, Val 1, Test 2
Splitting combination with labels: Infiltration, Effusion, Pneumothorax (49 samples)
  - Split: Train 39, Val 4, Test 6
Splitting combination with labels: Cardiomegaly, Emphysema, Mass, Effusion (13 samples)
  - Split: Train 10, Val 1, T

Training Random Forest models:   0%|          | 0/15 [00:00<?, ?class/s]


Training Random Forest for Fibrosis (1/15)...
  - Using class_weight = {0: 1.0, 1: np.float64(6.915880503144654)}


...

Generating predictions:   0%|          | 0/15 [00:00<?, ?class/s]


Calculating AUC-ROC scores...

Calculating detailed metrics...

Standard threshold (0.5) metrics:
              precision    recall  f1-score   support

           0       0.85      0.14      0.24       502
           1       0.98      0.38      0.55       268
           2       0.79      0.25      0.38       966
           3       0.74      0.31      0.43       915
           4       0.42      0.50      0.45       200
           5       0.72      0.50      0.59      1361
           6       0.89      0.15      0.25       510
           7       0.82      0.30      0.44       574
           8       0.83      0.27      0.41       616
           9       0.70      0.65      0.67      1730
          10       0.78      0.25      0.37       836
          11       0.81      0.40      0.54      1047
          12       0.76      0.66      0.71      1597
          13       0.82      0.33      0.48       842
          14       0.73      0.34      0.46       589

   micro avg       0.74      0.41  

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
<ipython-input-64-e16feef9947d>:9: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(x=metric_name, y='Label', data=sorted_df, palette='viridis')
<ipython-input-64-e16feef9947d>:9: FutureWarning: 

Passing `palette


Step 9: Saving results and model...

RANDOM FOREST IMPLEMENTATION COMPLETED SUCCESSFULLY!
Total execution time: 351.15 seconds (5.85 minutes)
Finished at: 2025-04-22 23:26:58
Random Forest implementation completed!


<Figure size 1400x1000 with 0 Axes>